# Component 3 - Phase 2: full-image check vs Konan (step 2.7)

The two finalists (Cellpose-SAM fine-tuned, StarDist fine-tuned) are trained on all 40 crops and run on the
**197 validation images**, with the classical method as reference. Primary criterion (pre-registered):
Spearman(mean cell area, Konan mean area); tie -> StarDist.

**Runtime > Change runtime type > L4 GPU.** Every cell can be re-run after a disconnect (finished work is skipped).

In [ ]:
# 1. Setup (mount Drive, pull latest code, copy images, set paths)
from google.colab import drive; drive.mount("/content/drive")
REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Research_Project/xai-cornea-grading"
!git -C "{REPO_DIR}" pull --ff-only
%run "{REPO_DIR}/ml/cellular/colab_setup.py"

In [ ]:
# 2. Packages (Colab only): Cellpose + StarDist, GPU check
!pip -q install "cellpose>=4.0,<5" tensorflow tf-keras stardist
!nvidia-smi -L

In [ ]:
# 3. Final Cellpose model on all 40 crops (~25 min on L4; kept on Drive, ~1.2 GB)
!python -m segmentation.train_final --model cellpose 2>&1 | tail -n 5

In [ ]:
# 4. Final StarDist model on all 40 crops (36 train + 4 for checkpoint/thresholds; ~10 min)
!python -m segmentation.train_final --model stardist 2>&1 | tail -n 5

In [ ]:
# 5. Segment the 197 validation images with each run (progress every 20 images)
!python -m segmentation.infer_full --run classical
!python -m segmentation.infer_full --run cellpose_finetune_flat 2>&1 | grep -v Warning
!python -m segmentation.infer_full --run stardist_finetune_flat 2>&1 | grep -E "images|done|Error|error"

In [ ]:
# 6. Comparison with Konan + pre-registered decision
!python -m segmentation.infer_full --compare

In [ ]:
# 7. Full-image overlays: one image per ECD band (classical | Cellpose | StarDist)
import pandas as pd
from IPython.display import Image, display
man = pd.read_csv("splits/manifest.csv")
val = man[(man["split"] == "val") & (man["status"] == "included")]
ids = [g.sample(1, random_state=0)["image_id"].iloc[0] for _, g in val.groupby("ecd_band")]
ARGS = " ".join('"' + i + '"' for i in ids)          # image IDs contain spaces
!python -m segmentation.infer_full --figure {ARGS}
for i in ids:
    display(Image(f"{OUTPUT_ROOT}/seg/full_val/figures/{i}.png"))